# LeetCode 560 — Subarray Sum Equals K bằng Python

Link đề: https://leetcode.com/problems/subarray-sum-equals-k/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def subarraySum(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: đây là bài "kinh điển" của kỹ thuật **prefix sum (tổng tiền tố) + hash map đếm tần suất**. Mảng có thể chứa **số âm và số 0**, nên cách "cửa sổ trượt" quen thuộc **không dùng được**. Notebook đi từ vét cạn `O(n^3)`, `O(n^2)`, rồi tới `O(n)` bằng hash map, và thêm một cách chia để trị `O(n log n)` để thấy cùng một bài có nhiều góc nhìn.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm **số đoạn con liên tiếp (subarray)** có tổng đúng bằng `k` — đếm **mọi** đoạn, kể cả hai đoạn có cùng giá trị nhưng khác vị trí.
2. Nắm các công cụ Python cần dùng: cắt lát `nums[i:j+1]`, `sum()`, `enumerate`, `itertools.accumulate`, `dict`, `collections.Counter`, `defaultdict`.
3. Hiểu **tổng tiền tố** `P[j] - P[i] = tổng nums[i..j-1]` và biến bài toán "đếm đoạn con có tổng k" thành "đếm cặp tiền tố hơn kém nhau đúng `k`".
4. Đi từ **vét cạn `O(n^3)`** → **cộng dồn `O(n^2)`** → **mảng tiền tố `O(n^2)`** → **hash map đếm tần suất `O(n)`** → **chia để trị `O(n log n)`**.
5. Tránh các lỗi kinh điển: dùng cửa sổ trượt khi có số âm, quên `cnt[0] = 1`, cập nhật map **trước** khi truy vấn, dùng `set` thay vì đếm, và `KeyError` khi tra khóa chưa có.

## 2. Đề bài & ví dụ

Cho một mảng số nguyên `nums` và một số nguyên `k`. Hãy trả về **tổng số đoạn con liên tiếp** có tổng các phần tử bằng `k`.

Đoạn con (subarray) là một dãy phần tử **liên tiếp và không rỗng** trong mảng.

**Ví dụ 1**

- Input: `nums = [1,1,1], k = 2`
- Output: `2`
- Giải thích: hai đoạn `[1,1]` (chỉ số 0–1) và `[1,1]` (chỉ số 1–2). Chúng trông giống nhau nhưng là **hai đoạn khác nhau** vì khác vị trí.

**Ví dụ 2**

- Input: `nums = [1,2,3], k = 3`
- Output: `2`
- Giải thích: `[1,2]` và `[3]`.

**Ràng buộc:**

- `1 <= nums.length <= 2 * 10^4`
- `-1000 <= nums[i] <= 1000`
- `-10^7 <= k <= 10^7`

Hai điều đáng chú ý: `nums[i]` có thể **âm** hoặc **bằng 0**; và `n` tới `20000` nên `O(n^2)` (khoảng `4 * 10^8` thao tác) rất sát giới hạn, `O(n)` mới an toàn.

In [1]:
from typing import List
from collections import Counter, defaultdict
from itertools import accumulate
import random

examples = [
    ([1, 1, 1], 2, 2),
    ([1, 2, 3], 3, 2),
]
for nums, k, expected in examples:
    print("nums =", nums, "| k =", k, "| kỳ vọng:", expected)

nums = [1, 1, 1] | k = 2 | kỳ vọng: 2
nums = [1, 2, 3] | k = 3 | kỳ vọng: 2


## 3. Kiến thức nền

### 3.1. Đoạn con liên tiếp và cắt lát `nums[i:j+1]`

Một đoạn con được xác định bởi **hai chỉ số** `i <= j`. Trong Python, `nums[i:j+1]` lấy các phần tử từ chỉ số `i` đến `j` (cắt lát **bao gồm đầu, loại trừ cuối**, nên phải `j+1`). Mảng `n` phần tử có đúng `n*(n+1)/2` đoạn con không rỗng (chọn cặp `i <= j`).

In [2]:
nums = [3, -1, 2, 1]
n = len(nums)
for i in range(n):
    for j in range(i, n):
        print(f"i={i} j={j} -> {nums[i:j+1]!s:14} tổng = {sum(nums[i:j+1])}")
print("số đoạn con:", n * (n + 1) // 2)

i=0 j=0 -> [3]            tổng = 3
i=0 j=1 -> [3, -1]        tổng = 2
i=0 j=2 -> [3, -1, 2]     tổng = 4
i=0 j=3 -> [3, -1, 2, 1]  tổng = 5
i=1 j=1 -> [-1]           tổng = -1
i=1 j=2 -> [-1, 2]        tổng = 1
i=1 j=3 -> [-1, 2, 1]     tổng = 2
i=2 j=2 -> [2]            tổng = 2
i=2 j=3 -> [2, 1]         tổng = 3
i=3 j=3 -> [1]            tổng = 1
số đoạn con: 10


### 3.2. `sum()` và cộng dồn bằng `accumulate`

`sum(list)` cộng mọi phần tử, tốn `O(độ dài)`. `itertools.accumulate(list)` cho ra **dãy tổng cộng dồn**: phần tử thứ `t` là tổng của `t+1` phần tử đầu. Đây chính là cách tạo **tổng tiền tố** gọn nhất. Ta tự cài tay cũng chỉ cần một vòng lặp.

In [3]:
nums = [3, -1, 2, 1]
print(list(accumulate(nums)))                 # [3, 2, 4, 5]
total = 0
manual = []
for x in nums:
    total += x
    manual.append(total)
print(manual, manual == list(accumulate(nums)))

[3, 2, 4, 5]
[3, 2, 4, 5] True


### 3.3. Tổng tiền tố (prefix sum) `P`

Định nghĩa `P[0] = 0` (tổng của **0 phần tử**, tức đoạn rỗng) và `P[t] = nums[0] + ... + nums[t-1]`. Khi đó:

> tổng của đoạn `nums[i..j]` (cả hai đầu) = `P[j+1] - P[i]`.

Lý do: `P[j+1]` là "tổng từ đầu tới `j`", trừ đi `P[i]` là "tổng từ đầu tới `i-1`" thì phần dư chính là đoạn `i..j`. Nhờ vậy, **tính tổng một đoạn chỉ còn `O(1)`** sau khi dựng `P` một lần.

In [4]:
nums = [3, -1, 2, 1]
P = [0]
for x in nums:
    P.append(P[-1] + x)
print("P =", P)
# kiểm tra công thức với mọi đoạn
for i in range(len(nums)):
    for j in range(i, len(nums)):
        assert sum(nums[i:j+1]) == P[j + 1] - P[i]
print("công thức P[j+1] - P[i] đúng với mọi đoạn")

P = [0, 3, 2, 4, 5]
công thức P[j+1] - P[i] đúng với mọi đoạn


### 3.4. `dict`, `Counter`, `defaultdict` — đếm tần suất

- `dict` là bảng băm: tra/ghi một khóa trung bình `O(1)`. Tra khóa **chưa có** bằng `d[key]` sẽ báo `KeyError`; dùng `d.get(key, 0)` để mặc định `0`.
- `Counter` là dict chuyên đếm; khóa chưa có trả `0` (và **không** tự tạo khóa khi chỉ đọc bằng `c[key]`).
- `defaultdict(int)` tự tạo khóa với giá trị `0` khi truy cập (kể cả khi chỉ đọc!).

In [5]:
d = {}
try:
    d[5]
except KeyError as e:
    print("dict: KeyError", e)
print("d.get(5, 0) =", d.get(5, 0))

c = Counter([1, 1, 2])
print(c, "| c[1] =", c[1], "| c[99] =", c[99], "| 99 in c?", 99 in c)

dd = defaultdict(int)
print("defaultdict đọc khóa 7 ->", dd[7], "| giờ có khóa 7?", 7 in dd)

dict: KeyError 5
d.get(5, 0) = 0
Counter({1: 2, 2: 1}) | c[1] = 2 | c[99] = 0 | 99 in c? False
defaultdict đọc khóa 7 -> 0 | giờ có khóa 7? True


## 4. Ý tưởng nền tảng / trực giác

Đoạn con `nums[i..j]` có tổng `k` khi và chỉ khi

`P[j+1] - P[i] = k`   hay   `P[i] = P[j+1] - k`.

Nghĩa là: **đứng ở một mốc tiền tố `P[j+1]`, ta hỏi "có bao nhiêu mốc tiền tố ĐỨNG TRƯỚC bằng `P[j+1] - k`?"** — mỗi mốc như vậy là một đoạn tổng `k` kết thúc đúng tại `j`.

Tính tay `nums = [1,2,3]`, `k = 3` → `P = [0, 1, 3, 6]`:

| mốc `P[b]` | cần tìm `P[b] - 3` trong các mốc trước | có? | đoạn |
|---|---|---|---|
| `P[1] = 1` | `-2` | không | - |
| `P[2] = 3` | `0` (là `P[0]`) | 1 | `[1,2]` |
| `P[3] = 6` | `3` (là `P[2]`) | 1 | `[3]` |

Tổng cộng `2` — đúng đáp án. Chú ý mốc `P[0] = 0` **phải có mặt từ đầu** vì nó đại diện cho "đoạn bắt đầu từ chỉ số 0". Đây là chi tiết dễ quên nhất của bài.

In [6]:
def pairs_by_prefix(nums, k):
    """Liệt kê mọi đoạn có tổng k bằng cách so cặp tiền tố (để quan sát)."""
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    found = []
    for b in range(1, len(P)):
        for a in range(b):
            if P[b] - P[a] == k:
                found.append((a, b - 1, nums[a:b]))   # đoạn nums[a..b-1]
    return P, found

P, found = pairs_by_prefix([1, 2, 3], 3)
print("P =", P)
for a, j, seg in found:
    print(f"P[{a}] ... đoạn nums[{a}..{j}] = {seg}")
P, found = pairs_by_prefix([1, 1, 1], 2)
print("P =", P, "| số đoạn:", len(found), found)

P = [0, 1, 3, 6]
P[0] ... đoạn nums[0..1] = [1, 2]
P[2] ... đoạn nums[2..2] = [3]
P = [0, 1, 2, 3] | số đoạn: 2 [(0, 1, [1, 1]), (1, 2, [1, 1])]


## 5. Cách 1 — Brute Force: thử mọi đoạn

### Ý tưởng

Thử **mọi cặp `(i, j)`** với `i <= j`, tính tổng đoạn rồi so với `k`.

- Bản ngây thơ nhất: dùng `sum(nums[i:j+1])` cho mỗi cặp → mỗi lần tính tốn `O(độ dài đoạn)` nên tổng `O(n^3)`.
- Cải tiến dễ: cố định `i`, cho `j` chạy sang phải và **cộng dồn** `s += nums[j]` (không tính lại từ đầu) → `O(n^2)`.

Cả hai đều **đúng** với mọi dấu của số. Ta dùng bản `O(n^3)` làm **oracle** (đối chứng) cho mọi test sau này vì nó ngây thơ đến mức khó sai.

In [7]:
def sub_cubic(nums, k):
    """O(n^3): thử mọi đoạn, tính tổng bằng sum() mỗi lần. Dùng làm oracle."""
    n = len(nums)
    cnt = 0
    for i in range(n):
        for j in range(i, n):
            if sum(nums[i:j + 1]) == k:
                cnt += 1
    return cnt


def sub_running(nums, k):
    """O(n^2): cố định i, cộng dồn s khi j tăng."""
    n = len(nums)
    cnt = 0
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            if s == k:
                cnt += 1
    return cnt


for nums, k, expected in examples:
    print(nums, k, "| cubic:", sub_cubic(nums, k), "| running:", sub_running(nums, k), "| kỳ vọng:", expected)
    assert sub_cubic(nums, k) == sub_running(nums, k) == expected

[1, 1, 1] 2 | cubic: 2 | running: 2 | kỳ vọng: 2
[1, 2, 3] 3 | cubic: 2 | running: 2 | kỳ vọng: 2


### Độ phức tạp và nút thắt

- `sub_cubic`: `O(n^3)` thời gian, `O(n)` bộ nhớ phụ (do cắt lát tạo list mới), `O(1)` nếu dùng `sum` trên chỉ số.
- `sub_running`: `O(n^2)` thời gian, `O(1)` bộ nhớ.

Với `n = 20000`, `O(n^2)` cỡ `2 * 10^8` phép (nửa số cặp) — trong Python gần như chắc chắn **quá thời gian**. Nút thắt: ta **lặp lại công việc** — với mỗi `i`, ta quét lại cả phần đuôi của mảng. Câu hỏi: có cách nào biết "có bao nhiêu `i` thỏa" **ngay tại `j`** mà không phải quét hết các `i`?

## 6. Ý tưởng thuật toán / bước đột phá

Từ mục 4, đoạn kết thúc tại `j` có tổng `k` khi tồn tại mốc trước `P[i] = P[j+1] - k`. Thay vì **quét** mọi `i` để tìm mốc đó, ta **ghi nhớ** mọi tổng tiền tố đã gặp cùng **số lần** gặp trong một hash map `cnt`:

- `cnt[v]` = số mốc tiền tố đã duyệt có giá trị bằng `v`.
- Tại mỗi vị trí với tổng tiền tố hiện tại `s`: cộng `cnt[s - k]` vào đáp án (mọi mốc trước bằng `s - k` đều cho một đoạn tổng `k`), rồi ghi nhận `cnt[s] += 1`.

Hai chi tiết sống còn:

1. **Khởi tạo `cnt[0] = 1`**: mốc rỗng `P[0] = 0` đã "có mặt" trước khi đọc phần tử nào.
2. **Truy vấn TRƯỚC, cập nhật SAU**: mốc hiện tại không được tự ghép với chính nó (khi `k = 0`, `s - k = s`, nếu cập nhật trước thì ta đếm nhầm đoạn rỗng).

Mỗi phần tử xử lý `O(1)` → tổng `O(n)`.

## 7. Các cách tiếp theo

Tổng cộng ta có 5 hàm thuộc 4 hướng nghĩ khác nhau: (1) vét cạn / cộng dồn (mục 5), (2) mảng tiền tố rồi so cặp, (3) hash map tiền tố, (4) chia để trị.

### 7.1. Cách 2 — Mảng tiền tố + so mọi cặp `O(n^2)`

**Giải thích từng đoạn.**

1. Dựng `P` với `P[0] = 0`.
2. Với mỗi mốc kết thúc `b` từ `1` đến `n`, duyệt mọi mốc bắt đầu `a < b`.
3. Nếu `P[b] - P[a] == k` thì đếm.

Mỗi tổng đoạn giờ tính trong `O(1)` nhưng ta vẫn xét `O(n^2)` cặp. Đây là bước đệm để hiểu rằng hash map sẽ thay vòng lặp trong (duyệt `a`) bằng một phép tra.

In [8]:
def sub_prefix_array(nums, k):
    n = len(nums)
    P = [0] * (n + 1)
    for t in range(n):
        P[t + 1] = P[t] + nums[t]
    cnt = 0
    for b in range(1, n + 1):
        for a in range(b):
            if P[b] - P[a] == k:
                cnt += 1
    return cnt


for nums, k, expected in examples:
    print(nums, k, "->", sub_prefix_array(nums, k))
    assert sub_prefix_array(nums, k) == expected

[1, 1, 1] 2 -> 2
[1, 2, 3] 3 -> 2


**Code nộp LeetCode (cách 2):**

```python
class Solution:
    def subarraySum(self, nums: List[int], k: int) -> int:
        n = len(nums)
        P = [0] * (n + 1)
        for t in range(n):
            P[t + 1] = P[t] + nums[t]
        cnt = 0
        for b in range(1, n + 1):
            for a in range(b):
                if P[b] - P[a] == k:
                    cnt += 1
        return cnt
```

**Độ phức tạp:** Time `O(n^2)`, Space `O(n)`. Cách này sẽ TLE với `n = 2 * 10^4` trên LeetCode.

### 7.2. Cách 3 — Hash map đếm tần suất tiền tố `O(n)` (cách chính)

**Giải thích từng đoạn.**

1. `cnt = {0: 1}`: tổng tiền tố rỗng đã xuất hiện một lần.
2. Duyệt từng `x`: `s += x` (tổng tiền tố hiện tại).
3. `ans += cnt.get(s - k, 0)`: số mốc trước có thể ghép thành đoạn tổng `k` kết thúc tại vị trí hiện tại.
4. `cnt[s] = cnt.get(s, 0) + 1`: ghi nhận mốc hiện tại **sau cùng**.

In [9]:
def sub_hash(nums, k):
    cnt = {0: 1}          # mốc rỗng P[0] = 0 đã xuất hiện 1 lần
    s = 0
    ans = 0
    for x in nums:
        s += x
        ans += cnt.get(s - k, 0)       # truy vấn TRƯỚC
        cnt[s] = cnt.get(s, 0) + 1     # cập nhật SAU
    return ans


for nums, k, expected in examples:
    print(nums, k, "->", sub_hash(nums, k))
    assert sub_hash(nums, k) == expected

[1, 1, 1] 2 -> 2
[1, 2, 3] 3 -> 2


**Code nộp LeetCode (cách 3, dùng `Counter` cho gọn):**

```python
from collections import Counter

class Solution:
    def subarraySum(self, nums: List[int], k: int) -> int:
        cnt = Counter({0: 1})
        ans = s = 0
        for x in nums:
            s += x
            ans += cnt[s - k]
            cnt[s] += 1
        return ans
```

Lưu ý `cnt[s - k]` với `Counter` chỉ **đọc** và trả `0` nếu chưa có (không tạo khóa thừa). Nếu thay bằng `defaultdict(int)` vẫn đúng nhưng mỗi lần đọc sẽ tạo thêm khóa `s - k` với giá trị `0`, tốn thêm bộ nhớ.

**Độ phức tạp:** Time `O(n)`, Space `O(n)` (tối đa `n + 1` khóa khác nhau).

### 7.3. Cách 4 — Chia để trị `O(n log n)`

**Ý tưởng khác hẳn.** Chia mảng làm hai nửa tại `mid`. Mọi đoạn con rơi vào đúng một trong ba loại: nằm **hoàn toàn trong nửa trái**, **hoàn toàn trong nửa phải**, hoặc **bắc cầu qua `mid`** (chứa cả `nums[mid-1]` và `nums[mid]`).

- Hai loại đầu: đệ quy.
- Loại bắc cầu: đoạn = (một **hậu tố** của nửa trái) + (một **tiền tố** của nửa phải). Gọi `L` là tổng các hậu tố trái, `R` là tổng các tiền tố phải; cần `l + r = k`. Đếm bằng một `Counter` các tổng hậu tố trái, rồi với mỗi `r` cộng `cntL[k - r]`.

Mỗi tầng đệ quy tốn `O(n)` cho phần bắc cầu, có `log n` tầng → `O(n log n)`. Cách này hữu ích khi bạn gặp biến thể mà prefix + map khó áp dụng (ví dụ cần thêm điều kiện khác trên đoạn).

In [10]:
def sub_divide(nums, k):
    def solve(lo, hi):                 # xét nums[lo:hi]
        if hi - lo == 0:
            return 0
        if hi - lo == 1:
            return 1 if nums[lo] == k else 0
        mid = (lo + hi) // 2
        total = solve(lo, mid) + solve(mid, hi)
        # đoạn bắc cầu: hậu tố của nửa trái + tiền tố của nửa phải
        left = Counter()
        s = 0
        for i in range(mid - 1, lo - 1, -1):    # hậu tố nums[i:mid]
            s += nums[i]
            left[s] += 1
        s = 0
        for j in range(mid, hi):                # tiền tố nums[mid:j+1]
            s += nums[j]
            total += left[k - s]
        return total
    return solve(0, len(nums))


for nums, k, expected in examples:
    print(nums, k, "->", sub_divide(nums, k))
    assert sub_divide(nums, k) == expected

[1, 1, 1] 2 -> 2
[1, 2, 3] 3 -> 2


**Code nộp LeetCode (cách 4):**

```python
from collections import Counter

class Solution:
    def subarraySum(self, nums: List[int], k: int) -> int:
        def solve(lo, hi):
            if hi - lo == 0:
                return 0
            if hi - lo == 1:
                return 1 if nums[lo] == k else 0
            mid = (lo + hi) // 2
            total = solve(lo, mid) + solve(mid, hi)
            left = Counter()
            s = 0
            for i in range(mid - 1, lo - 1, -1):
                s += nums[i]
                left[s] += 1
            s = 0
            for j in range(mid, hi):
                s += nums[j]
                total += left[k - s]
            return total
        return solve(0, len(nums))
```

**Độ phức tạp:** Time `O(n log n)`, Space `O(n)` (Counter ở mỗi tầng + độ sâu đệ quy `O(log n)`).

Bốn hướng cho cùng một đáp án; mục 12 sẽ đối chiếu chúng bằng `assert` trên rất nhiều input.

## 8. TRACE từng bước

Trace cách hash map (cách 3). Ta bắt đầu với ví dụ nhỏ `nums = [1,1,1], k = 2`.

| bước | `x` | `s` | cần `s-k` | `cnt[s-k]` | `ans` | `cnt` sau khi cập nhật |
|---|---|---|---|---|---|---|
| bắt đầu | - | 0 | - | - | 0 | `{0:1}` |
| 1 | 1 | 1 | -1 | 0 | 0 | `{0:1, 1:1}` |
| 2 | 1 | 2 | 0 | 1 | 1 | `{0:1, 1:1, 2:1}` |
| 3 | 1 | 3 | 1 | 1 | 2 | `{0:1, 1:1, 2:1, 3:1}` |

Ở bước 2, `cnt[0] = 1` chính là mốc rỗng: đoạn `[1,1]` (chỉ số 0–1). Ở bước 3, `cnt[1] = 1` là mốc sau phần tử đầu: đoạn `[1,1]` (chỉ số 1–2).

Hàm `trace_hash` dưới đây in từng bước và còn **liệt kê các đoạn** được đếm, để bạn kiểm chứng bằng mắt.

In [11]:
def trace_hash(nums, k):
    cnt = {0: 1}
    s = 0
    ans = 0
    # sums_at[v] = danh sách chỉ số 'mốc' (P index) có tổng v, để in ra đoạn
    where = {0: [0]}
    print(f"nums = {nums}, k = {k}")
    print(f"bắt đầu: s=0 cnt={cnt} ans=0")
    for idx, x in enumerate(nums):
        s += x
        need = s - k
        got = cnt.get(need, 0)
        ans += got
        segs = [nums[a:idx + 1] for a in where.get(need, [])]
        cnt[s] = cnt.get(s, 0) + 1
        where.setdefault(s, []).append(idx + 1)
        print(f"i={idx} x={x:>2} s={s:>3} cần s-k={need:>3} cnt[cần]={got} "
              f"-> ans={ans}  đoạn đếm thêm: {segs}  cnt={cnt}")
    return ans


print("kết quả:", trace_hash([1, 1, 1], 2))
print()
print("kết quả:", trace_hash([1, 2, 3], 3))

nums = [1, 1, 1], k = 2
bắt đầu: s=0 cnt={0: 1} ans=0
i=0 x= 1 s=  1 cần s-k= -1 cnt[cần]=0 -> ans=0  đoạn đếm thêm: []  cnt={0: 1, 1: 1}
i=1 x= 1 s=  2 cần s-k=  0 cnt[cần]=1 -> ans=1  đoạn đếm thêm: [[1, 1]]  cnt={0: 1, 1: 1, 2: 1}
i=2 x= 1 s=  3 cần s-k=  1 cnt[cần]=1 -> ans=2  đoạn đếm thêm: [[1, 1]]  cnt={0: 1, 1: 1, 2: 1, 3: 1}
kết quả: 2

nums = [1, 2, 3], k = 3
bắt đầu: s=0 cnt={0: 1} ans=0
i=0 x= 1 s=  1 cần s-k= -2 cnt[cần]=0 -> ans=0  đoạn đếm thêm: []  cnt={0: 1, 1: 1}
i=1 x= 2 s=  3 cần s-k=  0 cnt[cần]=1 -> ans=1  đoạn đếm thêm: [[1, 2]]  cnt={0: 1, 1: 1, 3: 1}
i=2 x= 3 s=  6 cần s-k=  3 cnt[cần]=1 -> ans=2  đoạn đếm thêm: [[3]]  cnt={0: 1, 1: 1, 3: 1, 6: 1}
kết quả: 2


Trace một ca có **số âm và số 0** để thấy tần suất lớn hơn 1: `nums = [1, -1, 0, 1, -1], k = 0`. Tổng tiền tố là `0, 1, 0, 0, 1, 0` — giá trị `0` xuất hiện 4 lần, `1` xuất hiện 2 lần. Số cặp bằng nhau: `C(4,2) + C(2,2) = 6 + 1 = 7`.

In [12]:
res = trace_hash([1, -1, 0, 1, -1], 0)
print("kết quả:", res)
assert res == sub_cubic([1, -1, 0, 1, -1], 0) == 7

nums = [1, -1, 0, 1, -1], k = 0
bắt đầu: s=0 cnt={0: 1} ans=0
i=0 x= 1 s=  1 cần s-k=  1 cnt[cần]=0 -> ans=0  đoạn đếm thêm: []  cnt={0: 1, 1: 1}
i=1 x=-1 s=  0 cần s-k=  0 cnt[cần]=1 -> ans=1  đoạn đếm thêm: [[1, -1]]  cnt={0: 2, 1: 1}
i=2 x= 0 s=  0 cần s-k=  0 cnt[cần]=2 -> ans=3  đoạn đếm thêm: [[1, -1, 0], [0]]  cnt={0: 3, 1: 1}
i=3 x= 1 s=  1 cần s-k=  1 cnt[cần]=1 -> ans=4  đoạn đếm thêm: [[-1, 0, 1]]  cnt={0: 3, 1: 2}
i=4 x=-1 s=  0 cần s-k=  0 cnt[cần]=3 -> ans=7  đoạn đếm thêm: [[1, -1, 0, 1, -1], [0, 1, -1], [1, -1]]  cnt={0: 4, 1: 2}
kết quả: 7


## 9. Code nộp LeetCode

Đây là cách tối ưu (`O(n)`), dùng đúng skeleton LeetCode. Trên LeetCode, `List` đã có sẵn; `Counter` được nhập sẵn trong môi trường của họ nhưng ta vẫn nên ghi `import` cho rõ.

```python
from collections import Counter
from typing import List

class Solution:
    def subarraySum(self, nums: List[int], k: int) -> int:
        cnt = Counter({0: 1})   # mốc rỗng: tổng 0 đã xuất hiện 1 lần
        ans = s = 0
        for x in nums:
            s += x                # tổng tiền tố hiện tại
            ans += cnt[s - k]     # truy vấn trước
            cnt[s] += 1           # cập nhật sau
        return ans
```

In [13]:
class Solution:
    def subarraySum(self, nums: List[int], k: int) -> int:
        cnt = Counter({0: 1})
        ans = s = 0
        for x in nums:
            s += x
            ans += cnt[s - k]
            cnt[s] += 1
        return ans


sol = Solution()
for nums, k, expected in examples:
    got = sol.subarraySum(nums, k)
    print(nums, k, "->", got, "| kỳ vọng", expected)
    assert got == expected

[1, 1, 1] 2 -> 2 | kỳ vọng 2
[1, 2, 3] 3 -> 2 | kỳ vọng 2


## 10. Vì sao đúng

**Phát biểu.** Với mỗi vị trí `j`, vòng lặp cộng vào `ans` đúng **số đoạn con có tổng `k` kết thúc tại `j`**.

**Chứng minh ngắn.**

1. Đoạn `nums[i..j]` có tổng `k` ⇔ `P[j+1] - P[i] = k` ⇔ `P[i] = P[j+1] - k`, với `0 <= i <= j`.
2. Khi xử lý phần tử `nums[j]`, biến `s = P[j+1]`; map `cnt` đang chứa **đúng các mốc `P[0], P[1], ..., P[j]`** (nhờ khởi tạo `P[0]` và cập nhật `P[t]` sau mỗi lần truy vấn ở bước `t-1`). Chính vì "truy vấn trước, cập nhật sau" nên `P[j+1]` chưa có trong map, đảm bảo `i <= j` (đoạn không rỗng, `i != j+1`).
3. Vậy `cnt[s - k]` = số chỉ số `i in [0, j]` thỏa `P[i] = s - k` = số đoạn tổng `k` kết thúc tại `j`.
4. Mỗi đoạn kết thúc tại đúng một `j`, nên cộng qua mọi `j` đếm **mỗi đoạn đúng một lần**.

**Vì sao không cần dấu của số?** Chứng minh không dùng bất kỳ tính đơn điệu nào của `P`; chỉ dùng phép trừ. Do đó số âm, số 0 đều hợp lệ — khác hẳn cửa sổ trượt cần `P` tăng.

**Vì sao `cnt[0] = 1` là đúng?** `P[0]` là một mốc thật (`i = 0` nghĩa là đoạn bắt đầu từ phần tử đầu tiên).

## 11. Time/Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| `sub_cubic` | `O(n^3)` | `O(n)` | oracle |
| `sub_running` | `O(n^2)` | `O(1)` | TLE với `n = 2 * 10^4` |
| `sub_prefix_array` | `O(n^2)` | `O(n)` | TLE |
| `sub_hash` / `Solution` | `O(n)` | `O(n)` | cách nộp |
| `sub_divide` | `O(n log n)` | `O(n)` | hướng khác, đủ nhanh |

Với hash map: mỗi phần tử một lần tra + một lần ghi trung bình `O(1)`, nên `O(n)`. Bộ nhớ tối đa `n + 1` khóa.

Một lưu ý nhỏ về giá trị lớn: tổng tiền tố tối đa `2 * 10^4 * 1000 = 2 * 10^7`; Python không tràn số nên không cần lo, còn trong C++/Java vẫn nằm gọn trong `int`.

## 12. Bộ test

Chiến lược: (a) ví dụ của đề, (b) edge case, (c) phản ví dụ cho các cách giải sai, (d) đối chiếu **tất cả các cách** với oracle `sub_cubic` trên nhiều input ngẫu nhiên có seed, (e) test kích thước lớn đúng ràng buộc.

### 12.1. Ví dụ của đề và edge case

In [14]:
ALL = {
    "running": sub_running,
    "prefix_array": sub_prefix_array,
    "hash": sub_hash,
    "divide": sub_divide,
    "solution": lambda nums, k: Solution().subarraySum(nums, k),
}

cases = [
    # (nums, k, expected)
    ([1, 1, 1], 2, 2),            # ví dụ 1
    ([1, 2, 3], 3, 2),            # ví dụ 2
    ([1], 1, 1),                  # một phần tử khớp
    ([1], 0, 0),                  # một phần tử không khớp
    ([0], 0, 1),                  # chính phần tử 0
    ([0, 0, 0], 0, 6),            # mọi đoạn đều tổng 0: n(n+1)/2
    ([1, -1, 0], 0, 3),           # [1,-1], [0], [1,-1,0]
    ([-1, -1, 1], 0, 1),          # [-1,1]
    ([3, 4, 7, 2, -3, 1, 4, 2], 7, 4),
    ([1, 2, 3], 100, 0),          # k quá lớn
    ([-5, -5], -10, 1),           # k âm
    ([1000] * 5, 2000, 4),        # giá trị lớn, nhiều đoạn trùng
]
for nums, k, expected in cases:
    assert sub_cubic(nums, k) == expected, (nums, k, sub_cubic(nums, k))
    for name, fn in ALL.items():
        got = fn(nums, k)
        assert got == expected, (name, nums, k, got, expected)
print("tất cả", len(cases), "ca cố định đều đúng với", len(ALL), "cách")

tất cả 12 ca cố định đều đúng với 5 cách


### 12.2. Phản ví dụ cho cửa sổ trượt (sai khi có số âm/0)

Cửa sổ trượt "tăng `r`, nếu tổng > k thì co `l`" chỉ đúng khi mọi phần tử **dương**. Dưới đây là hàm sai và ca làm lộ lỗi.

In [15]:
def wrong_sliding_window(nums, k):
    l = s = cnt = 0
    for r, x in enumerate(nums):
        s += x
        while s > k and l <= r:
            s -= nums[l]
            l += 1
        if s == k:
            cnt += 1
    return cnt


bad_cases = [([1, -1, 1], 1), ([1, 2, 3], 3), ([0, 0, 0], 0), ([2, -1, 2], 3)]
for nums, k in bad_cases:
    ok = wrong_sliding_window(nums, k) == sub_cubic(nums, k)
    print(nums, k, "| sliding window:", wrong_sliding_window(nums, k), "| đúng:", sub_cubic(nums, k), "| khớp?", ok)
assert wrong_sliding_window([1, -1, 1], 1) != sub_cubic([1, -1, 1], 1)
assert wrong_sliding_window([0, 0, 0], 0) != sub_cubic([0, 0, 0], 0)

[1, -1, 1] 1 | sliding window: 2 | đúng: 3 | khớp? False
[1, 2, 3] 3 | sliding window: 2 | đúng: 2 | khớp? True
[0, 0, 0] 0 | sliding window: 3 | đúng: 6 | khớp? False
[2, -1, 2] 3 | sliding window: 1 | đúng: 1 | khớp? True


### 12.3. Đối chiếu ngẫu nhiên có seed (input nhỏ, nhiều số 0 và số âm)

Dùng giá trị nhỏ `[-3, 3]` để các tổng **trùng nhau nhiều** — đây là chỗ các lỗi về đếm tần suất lộ ra.

In [16]:
rng = random.Random(560)
checked = 0
for _ in range(1500):
    n = rng.randint(1, 14)
    nums = [rng.randint(-3, 3) for _ in range(n)]
    k = rng.randint(-6, 6)
    expected = sub_cubic(nums, k)
    for name, fn in ALL.items():
        assert fn(nums, k) == expected, (name, nums, k)
    checked += 1
print("đã đối chiếu", checked, "input ngẫu nhiên nhỏ")

đã đối chiếu 1500 input ngẫu nhiên nhỏ


### 12.4. Ngẫu nhiên đúng ràng buộc (giá trị tới 1000), `k` lấy từ một đoạn thật

Nếu chọn `k` hoàn toàn ngẫu nhiên thì đa số ca có đáp án `0` (kém giá trị kiểm thử). Ta chọn `k` bằng tổng của một đoạn ngẫu nhiên để chắc chắn có đáp án dương.

In [17]:
rng = random.Random(2024)
for _ in range(60):
    n = rng.randint(1, 120)
    nums = [rng.randint(-1000, 1000) for _ in range(n)]
    i = rng.randrange(n)
    j = rng.randrange(i, n)
    k = sum(nums[i:j + 1])
    expected = sub_running(nums, k)           # O(n^2), nhanh với n <= 120
    assert expected >= 1
    for name in ("prefix_array", "hash", "divide", "solution"):
        assert ALL[name](nums, k) == expected, (name, nums, k)
print("60 input kích thước tới 120 đều khớp")

# mảng toàn số nhỏ để có nhiều đoạn cùng tổng
rng = random.Random(7)
for _ in range(60):
    n = rng.randint(50, 200)
    nums = [rng.choice([-1, 0, 1]) for _ in range(n)]
    k = rng.randint(-3, 3)
    expected = sub_running(nums, k)
    for name in ("prefix_array", "hash", "divide", "solution"):
        assert ALL[name](nums, k) == expected
print("60 input {-1,0,1} đều khớp")

60 input kích thước tới 120 đều khớp


60 input {-1,0,1} đều khớp


### 12.5. Test kích thước lớn (đúng ràng buộc `n = 2 * 10^4`)

Chỉ chạy các cách nhanh (`hash`, `divide`). Ca "toàn số 0, `k = 0`" cho đáp án rất lớn `n(n+1)/2` — kiểm tra luôn việc đếm không bị tràn hay sai.

In [18]:
import time

n = 20000
zeros = [0] * n
t = time.perf_counter()
r1 = sub_hash(zeros, 0)
r2 = sub_divide(zeros, 0)
print("toàn 0: hash =", r1, "| divide =", r2, "| n(n+1)/2 =", n * (n + 1) // 2)
assert r1 == r2 == n * (n + 1) // 2

rng = random.Random(99)
big = [rng.randint(-1000, 1000) for _ in range(n)]
kk = sum(big[100:5000])
a, b, c = sub_hash(big, kk), sub_divide(big, kk), Solution().subarraySum(big, kk)
print("ngẫu nhiên lớn: hash =", a, "| divide =", b, "| solution =", c)
assert a == b == c and a >= 1

alt = [1, -1] * (n // 2)
assert sub_hash(alt, 0) == sub_divide(alt, 0) == Solution().subarraySum(alt, 0)
print("xen kẽ 1,-1 với k=0:", sub_hash(alt, 0))
print("thời gian các test lớn: chỉ vài giây trở xuống ->", round(time.perf_counter() - t, 2), "s")

toàn 0: hash = 200010000 | divide = 200010000 | n(n+1)/2 = 200010000


ngẫu nhiên lớn: hash = 1160 | divide = 1160 | solution = 1160


xen kẽ 1,-1 với k=0: 100000000
thời gian các test lớn: chỉ vài giây trở xuống -> 0.23 s


### 12.6. Đối chiếu `O(n^2)` trên cỡ trung bình

Một lần so `sub_running` với `sub_hash` ở `n = 1500` để chắc chắn không có lỗi "chỉ lộ ra khi mảng dài".

In [19]:
rng = random.Random(1500)
mid_nums = [rng.randint(-20, 20) for _ in range(1500)]
for k in (0, 1, -7, 25, 100):
    assert sub_running(mid_nums, k) == sub_hash(mid_nums, k) == sub_divide(mid_nums, k)
print("n = 1500, 5 giá trị k: khớp")

n = 1500, 5 giá trị k: khớp


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng cửa sổ trượt (two pointers) khi có số âm hoặc 0

**Code sai:** xem `wrong_sliding_window` ở mục 12.2. **Vì sao sai:** cửa sổ trượt dựa vào "thêm phần tử làm tổng tăng, bớt phần tử làm tổng giảm". Với số âm/0, co `l` khi `s > k` có thể bỏ lỡ đoạn đúng, vì **thêm phần tử phải nữa có thể làm tổng giảm lại về `k`**. Ví dụ `[1,-1,1]`, `k=1` có `3` đoạn nhưng cửa sổ trượt chỉ đếm `2`.

**Code đúng:** dùng prefix sum + hash map (`sub_hash`). Cửa sổ trượt chỉ dùng được khi **toàn số không âm** (và cần cẩn thận với số 0).

### Lỗi 2 — Quên khởi tạo `cnt[0] = 1`

Mất mọi đoạn bắt đầu từ chỉ số 0.

In [20]:
def wrong_no_zero_seed(nums, k):
    cnt = Counter()                 # SAI: thiếu {0: 1}
    ans = s = 0
    for x in nums:
        s += x
        ans += cnt[s - k]
        cnt[s] += 1
    return ans


print("sai :", wrong_no_zero_seed([1, 1, 1], 2), "| đúng:", sub_hash([1, 1, 1], 2))
print("sai :", wrong_no_zero_seed([3], 3), "| đúng:", sub_hash([3], 3))
assert wrong_no_zero_seed([1, 1, 1], 2) == 1
assert sub_hash([1, 1, 1], 2) == 2

sai : 1 | đúng: 2
sai : 0 | đúng: 1


**Vì sao sai:** đoạn `[1,1]` ở chỉ số 0–1 cần mốc rỗng `P[0] = 0`; không có nó thì không đếm được. **Cách đúng:** `Counter({0: 1})` hoặc `cnt = {0: 1}`.

### Lỗi 3 — Cập nhật map TRƯỚC khi truy vấn

In [21]:
def wrong_update_first(nums, k):
    cnt = Counter({0: 1})
    ans = s = 0
    for x in nums:
        s += x
        cnt[s] += 1                 # SAI: cập nhật trước
        ans += cnt[s - k]
    return ans


print("sai :", wrong_update_first([1], 0), "| đúng:", sub_hash([1], 0))
print("sai :", wrong_update_first([1, 2], 0), "| đúng:", sub_hash([1, 2], 0))
assert wrong_update_first([1], 0) == 1 and sub_hash([1], 0) == 0

sai : 1 | đúng: 0
sai : 2 | đúng: 0


**Vì sao sai:** khi `k = 0` thì `s - k = s`; mốc hiện tại tự khớp với chính nó, tức đếm một **đoạn rỗng** không hợp lệ. **Cách đúng:** truy vấn trước, cập nhật sau.

### Lỗi 4 — Chỉ nhớ "đã thấy hay chưa" (set) thay vì **đếm số lần**

In [22]:
def wrong_set(nums, k):
    seen = {0}
    ans = s = 0
    for x in nums:
        s += x
        if s - k in seen:           # SAI: chỉ cộng 1 dù có nhiều mốc trùng
            ans += 1
        seen.add(s)
    return ans


print("sai :", wrong_set([1, -1, 1, -1], 0), "| đúng:", sub_hash([1, -1, 1, -1], 0))
print("sai :", wrong_set([0, 0, 0], 0), "| đúng:", sub_hash([0, 0, 0], 0))
assert wrong_set([1, -1, 1, -1], 0) == 3 and sub_hash([1, -1, 1, -1], 0) == 4
assert wrong_set([0, 0, 0], 0) == 3 and sub_hash([0, 0, 0], 0) == 6

sai : 3 | đúng: 4
sai : 3 | đúng: 6


**Vì sao sai:** nếu giá trị `s - k` đã xuất hiện **3 lần** làm mốc, thì có **3 đoạn** kết thúc ở đây, không phải 1. Bài này hỏi **đếm số đoạn**, không phải "có tồn tại không". **Cách đúng:** lưu `Counter`/dict giá trị → số lần.

### Lỗi 5 — Tra khóa chưa có bằng `dict[key]` → `KeyError`

In [23]:
def wrong_keyerror(nums, k):
    cnt = {0: 1}
    ans = s = 0
    for x in nums:
        s += x
        ans += cnt[s - k]           # SAI: s-k có thể chưa tồn tại
        cnt[s] = cnt.get(s, 0) + 1
    return ans


try:
    wrong_keyerror([1, 1, 1], 2)
except KeyError as e:
    print("KeyError với khóa", e)

KeyError với khóa -1


**Cách đúng:** `cnt.get(s - k, 0)`, hoặc dùng `Counter` (đọc khóa lạ trả `0`).

### Lỗi 6 (bonus) — Nhầm "đoạn" với "giá trị": đếm đoạn theo nội dung thay vì vị trí

Hai đoạn `[1,1]` ở vị trí khác nhau là **hai** đoạn. Nếu bạn bỏ trùng bằng `set` các tuple nội dung thì sẽ thiếu. Bài này luôn đếm theo **cặp chỉ số**.

In [24]:
nums, k = [1, 1, 1], 2
contents = set()
for i in range(len(nums)):
    for j in range(i, len(nums)):
        if sum(nums[i:j + 1]) == k:
            contents.add(tuple(nums[i:j + 1]))
print("đếm theo nội dung (sai):", len(contents), "| đếm theo vị trí (đúng):", sub_hash(nums, k))
assert len(contents) == 1 and sub_hash(nums, k) == 2

đếm theo nội dung (sai): 1 | đếm theo vị trí (đúng): 2


## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra kỹ thuật "prefix sum + hash map":**

- Đề hỏi **đếm số** (hoặc độ dài lớn nhất/nhỏ nhất) của **đoạn con liên tiếp** với tổng/điều kiện tính được từ tổng.
- Mảng có **số âm** → cửa sổ trượt thất bại → nghĩ tới prefix + map.
- Có thể viết điều kiện dạng `P[j] - P[i] = const` hoặc `(P[j] - P[i]) % K = 0`.

**Công thức cố định:**

```
cnt = {0: 1}; s = 0
for x in nums:
    s += x
    ans += cnt[s - k]     # truy vấn trước
    cnt[s] += 1           # cập nhật sau
```

**Mẹo nhớ:**

1. "Mốc rỗng có sẵn" → `{0: 1}`.
2. "Hỏi trước, ghi sau" → tránh tự ghép với chính mình.
3. Cần **đếm** thì map lưu **số lần**; cần **độ dài lớn nhất** thì map lưu **chỉ số xuất hiện đầu tiên** (và không ghi đè); cần "có hay không" thì set.
4. Khi `k` nằm trong phép trừ (`s - k`) nhớ xét dấu: `k` âm vẫn đúng không cần xử lý riêng.
5. Nếu đề cho toàn số **dương** và cần tối ưu bộ nhớ → cửa sổ trượt `O(1)` bộ nhớ là một lựa chọn; ở bài này, vì có số âm nên không được.

## 15. Biến thể

Cùng một khung "prefix + map" giải được nhiều bài nổi tiếng. Dưới đây là các biến thể kèm code ngắn đã kiểm chứng.

### 15.1. Đoạn con **dài nhất** có tổng `k` (LeetCode 325, Premium)

Thay vì đếm, ta lưu **chỉ số đầu tiên** mỗi tổng tiền tố xuất hiện; không ghi đè khi gặp lại (để đoạn dài nhất).

In [25]:
def longest_sum_k(nums, k):
    first = {0: -1}                  # tổng 0 xuất hiện "trước" chỉ số 0
    s = 0
    best = 0
    for i, x in enumerate(nums):
        s += x
        if s - k in first:
            best = max(best, i - first[s - k])
        if s not in first:           # chỉ ghi lần đầu
            first[s] = i
    return best


assert longest_sum_k([1, -1, 5, -2, 3], 3) == 4
assert longest_sum_k([-2, -1, 2, 1], 1) == 2
assert longest_sum_k([1, 2, 3], 100) == 0
print(longest_sum_k([1, -1, 5, -2, 3], 3), longest_sum_k([-2, -1, 2, 1], 1))

4 2


### 15.2. Đoạn con có tổng chia hết cho `K` (LeetCode 974)

Điều kiện `(P[j] - P[i]) % K == 0` ⇔ `P[j] % K == P[i] % K`. Ta đếm theo **số dư** của tổng tiền tố. Python cho `%` với `K > 0` luôn ra kết quả không âm nên xử lý số âm tự nhiên.

In [26]:
def count_div_k(nums, K):
    cnt = Counter({0: 1})
    ans = s = 0
    for x in nums:
        s = (s + x) % K
        ans += cnt[s]
        cnt[s] += 1
    return ans


print(count_div_k([4, 5, 0, -2, -3, 1], 5))
assert count_div_k([4, 5, 0, -2, -3, 1], 5) == 7
assert count_div_k([5], 9) == 0

7


### 15.3. Mảng nhị phân có số `0` và `1` bằng nhau dài nhất (LeetCode 525)

Đổi mọi `0` thành `-1`; khi đó "số 0 bằng số 1" ⇔ tổng đoạn bằng `0`. Dùng lại `longest_sum_k(..., 0)`.

In [27]:
def longest_balanced(bits):
    return longest_sum_k([1 if b else -1 for b in bits], 0)


assert longest_balanced([0, 1]) == 2
assert longest_balanced([0, 1, 0]) == 2
assert longest_balanced([1, 1, 1]) == 0
print(longest_balanced([0, 0, 1, 0, 0, 0, 1, 1]))

6


### 15.4. Mảng nhị phân, đếm đoạn có tổng `goal` (LeetCode 930) bằng cửa sổ trượt

Khi mọi phần tử **không âm**, ta có cách `O(1)` bộ nhớ: `số đoạn tổng đúng goal = atMost(goal) - atMost(goal - 1)`, trong đó `atMost(g)` đếm đoạn có tổng `<= g` bằng cửa sổ trượt.

In [28]:
def count_binary_goal(nums, goal):
    def at_most(g):
        if g < 0:
            return 0
        l = s = res = 0
        for r, x in enumerate(nums):
            s += x
            while s > g:
                s -= nums[l]
                l += 1
            res += r - l + 1      # mọi đoạn kết thúc tại r, bắt đầu từ l..r
        return res
    return at_most(goal) - at_most(goal - 1)


assert count_binary_goal([1, 0, 1, 0, 1], 2) == 4
assert count_binary_goal([0, 0, 0, 0, 0], 0) == 15
rng = random.Random(930)
for _ in range(200):
    arr = [rng.randint(0, 1) for _ in range(rng.randint(1, 15))]
    g = rng.randint(0, 4)
    assert count_binary_goal(arr, g) == sub_hash(arr, g)
print("930 khớp với sub_hash trên 200 input")

930 khớp với sub_hash trên 200 input


### 15.5. Tích nhỏ hơn `K` (LeetCode 713) — chỉ dùng được cửa sổ trượt vì mọi số dương

Một ví dụ ngược lại: khi phép toán là **tích** của số nguyên dương thì tích tăng đơn điệu, cửa sổ trượt hợp lệ.

In [29]:
def count_product_less(nums, K):
    if K <= 1:
        return 0
    l = 0
    prod = 1
    res = 0
    for r, x in enumerate(nums):
        prod *= x
        while prod >= K:
            prod //= nums[l]
            l += 1
        res += r - l + 1
    return res


assert count_product_less([10, 5, 2, 6], 100) == 8
assert count_product_less([1, 2, 3], 0) == 0
print(count_product_less([10, 5, 2, 6], 100))

8


Các biến thể khác để bạn tự nghĩ thêm: đếm đoạn có tổng trong khoảng `[lo, hi]` (cần BIT/merge sort trên tiền tố — LeetCode 327), ma trận con có tổng `target` (LeetCode 1074: nén hàng rồi áp dụng bài này), đoạn có số lẻ đúng `k` (LeetCode 1248: đổi số lẻ thành `1`, số chẵn thành `0`).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn `O(n^3)` | thử mọi đoạn, `sum` từng đoạn | `O(n^3)` | `O(n)` | oracle kiểm thử, `n` rất nhỏ |
| 1b. Cộng dồn `O(n^2)` | cố định `i`, `j` chạy, cộng dồn | `O(n^2)` | `O(1)` | `n` vài nghìn, tiết kiệm bộ nhớ |
| 2. Mảng tiền tố + cặp | `P[b] - P[a] == k` cho mọi cặp | `O(n^2)` | `O(n)` | bước đệm để hiểu prefix |
| 3. Hash map tiền tố | `cnt[s - k]`, truy vấn trước ghi sau | `O(n)` | `O(n)` | **cách nộp**, chạy tốt mọi ràng buộc |
| 4. Chia để trị | đếm đoạn bắc cầu qua `mid` bằng Counter | `O(n log n)` | `O(n)` | khi cần thêm điều kiện khó cho map, hoặc luyện tư duy D&C |

Khuyến nghị: luôn chọn **cách 3** khi nộp bài; dùng `sub_cubic` làm oracle khi tự viết biến thể.

Đối chiếu nhanh cả 5 hàm trên một vài input cuối cùng:

In [30]:
rng = random.Random(16)
for _ in range(300):
    nums = [rng.randint(-4, 4) for _ in range(rng.randint(1, 25))]
    k = rng.randint(-8, 8)
    res = {name: fn(nums, k) for name, fn in ALL.items()}
    res["cubic"] = sub_cubic(nums, k)
    assert len(set(res.values())) == 1, (nums, k, res)
print("300 input: 6 hàm cho cùng đáp án")

300 input: 6 hàm cho cùng đáp án


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(list(accumulate([3, -1, 2, 1])))
print(sub_hash([3, -1, 2, 1], 3))
print(sub_hash([0, 0, 0], 0))
```

In [31]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay bằng bảng hash map (dễ-vừa)

Với `nums = [1, 2, 1, 2, 1]` và `k = 3`, hãy lập bảng gồm: `x`, `s`, `s - k`, `cnt[s - k]`, `ans` ở mỗi bước. Đáp án cuối cùng là bao nhiêu, và đó là những đoạn nào?

In [32]:
# Bài 2: điền kết quả tính tay
hand_answer = None        # tổng số đoạn
hand_segments = None      # danh sách các đoạn dưới dạng list các list, ví dụ [[1, 2], ...]

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo khuôn mẫu "truy vấn trước, cập nhật sau".

In [33]:
def exercise_3_sub(nums: List[int], k: int) -> int:
    cnt = {0: 1}
    s = 0
    ans = 0
    for x in nums:
        # TODO 1: cộng x vào s
        # TODO 2: cộng số mốc trước có giá trị s - k vào ans (dùng cnt.get(..., 0))
        # TODO 3: ghi nhận mốc s vào cnt (tăng số lần lên 1)
        pass
    return None


# assert exercise_3_sub([1, 1, 1], 2) == 2
# assert exercise_3_sub([1, 2, 3], 3) == 2
# assert exercise_3_sub([0, 0, 0], 0) == 6

### Bài 4 — Tự viết lại (vừa-khó): đoạn dài nhất có tổng `k`

Không nhìn lại mục 15.1, hãy tự viết `exercise_4_longest(nums, k)` trả về **độ dài** đoạn con dài nhất có tổng đúng `k` (không có thì trả `0`).

```python
exercise_4_longest([1, -1, 5, -2, 3], 3)  -> 4
exercise_4_longest([-2, -1, 2, 1], 1)     -> 2
exercise_4_longest([1, 2, 3], 100)        -> 0
```

In [34]:
def exercise_4_longest(nums: List[int], k: int) -> int:
    # TODO
    pass


# assert exercise_4_longest([1, -1, 5, -2, 3], 3) == 4
# assert exercise_4_longest([-2, -1, 2, 1], 1) == 2
# assert exercise_4_longest([1, 2, 3], 100) == 0

### Bài 5 — Mini challenge: đếm đoạn có tổng chia hết cho `K` (khó)

Viết `exercise_5_div(nums, K)` đếm số đoạn con (không rỗng) có tổng chia hết cho `K` (`K >= 1`), có thể có số âm.

```python
exercise_5_div([4, 5, 0, -2, -3, 1], 5) -> 7
exercise_5_div([5], 9)                  -> 0
exercise_5_div([-1, 2, 9], 2)           -> 2
```

Gợi ý hướng nghĩ: hai mốc tiền tố cho đoạn chia hết cho `K` khi chúng **cùng số dư** khi chia `K`.

In [35]:
def exercise_5_div(nums: List[int], K: int) -> int:
    # TODO
    pass


# assert exercise_5_div([4, 5, 0, -2, -3, 1], 5) == 7
# assert exercise_5_div([5], 9) == 0
# assert exercise_5_div([-1, 2, 9], 2) == 2

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `accumulate` cho tổng cộng dồn; dòng 2 đếm đoạn tổng 3.
- Mức 2: các tổng tiền tố (kèm 0) là `0, 3, 2, 4, 5`; tìm cặp hơn kém đúng 3.
- Mức 3: dòng 1 là `[3, 2, 4, 5]`; dòng 2 là `2` (đoạn `[3]` và `[2,1]`); dòng 3 là `6`.

**Bài 2**
- Mức 1: tổng tiền tố lần lượt `1, 3, 4, 6, 7` (kèm mốc đầu `0`).
- Mức 2: ở mỗi bước, tra `s - 3` trong các mốc đã có `{0, 1, 3, 4, ...}`.
- Mức 3: có 4 bước tra thành công: `s=3` (cần 0), `s=4` (cần 1), `s=6` (cần 3), `s=7` (cần 4) → `4` đoạn: `[1,2]`, `[2,1]`, `[1,2]`, `[2,1]`.

**Bài 3**
- Mức 1: ba việc đúng thứ tự: cộng `s`, truy vấn, cập nhật.
- Mức 2: `s += x`; `ans += cnt.get(s - k, 0)`; `cnt[s] = cnt.get(s, 0) + 1`.
- Mức 3: nhớ `return ans` (không phải `None`).

**Bài 4**
- Mức 1: map lưu **chỉ số xuất hiện đầu tiên** của mỗi tổng tiền tố, khởi tạo `{0: -1}`.
- Mức 2: tại `i`, nếu `s - k` có trong map thì độ dài ứng viên `i - first[s - k]`.
- Mức 3: chỉ ghi `first[s] = i` khi `s` **chưa** có trong map; lấy `max` các ứng viên.

**Bài 5**
- Mức 1: thay "tổng tiền tố" bằng "tổng tiền tố **mod K**".
- Mức 2: `cnt = Counter({0: 1})`; `s = (s + x) % K`; `ans += cnt[s]`; `cnt[s] += 1`.
- Mức 3: không cần xử lý số âm riêng vì `%` của Python trả số dư không âm khi `K > 0`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [36]:
print(list(accumulate([3, -1, 2, 1])))
print(sub_hash([3, -1, 2, 1], 3))
print(sub_hash([0, 0, 0], 0))

assert list(accumulate([3, -1, 2, 1])) == [3, 2, 4, 5]
assert sub_hash([3, -1, 2, 1], 3) == 2
assert sub_hash([0, 0, 0], 0) == 6

[3, 2, 4, 5]
2
6


### Đáp án Bài 2

| bước | `x` | `s` | `s - k` | `cnt[s-k]` | `ans` | `cnt` sau |
|---|---|---|---|---|---|---|
| bắt đầu | - | 0 | - | - | 0 | `{0:1}` |
| 1 | 1 | 1 | -2 | 0 | 0 | `{0:1, 1:1}` |
| 2 | 2 | 3 | 0 | 1 | 1 | `{0:1, 1:1, 3:1}` |
| 3 | 1 | 4 | 1 | 1 | 2 | `{..., 4:1}` |
| 4 | 2 | 6 | 3 | 1 | 3 | `{..., 6:1}` |
| 5 | 1 | 7 | 4 | 1 | 4 | `{..., 7:1}` |

Đáp án `4`: `[1,2]` (0–1), `[2,1]` (1–2), `[1,2]` (2–3), `[2,1]` (3–4).

In [37]:
ans_2 = trace_hash([1, 2, 1, 2, 1], 3)
assert ans_2 == 4

segments = [[1, 2], [2, 1], [1, 2], [2, 1]]
real = [[1, 2, 1, 2, 1][i:j + 1] for i in range(5) for j in range(i, 5) if sum([1, 2, 1, 2, 1][i:j + 1]) == 3]
assert real == segments
print("Bài 2 OK:", len(real), real)

nums = [1, 2, 1, 2, 1], k = 3
bắt đầu: s=0 cnt={0: 1} ans=0
i=0 x= 1 s=  1 cần s-k= -2 cnt[cần]=0 -> ans=0  đoạn đếm thêm: []  cnt={0: 1, 1: 1}
i=1 x= 2 s=  3 cần s-k=  0 cnt[cần]=1 -> ans=1  đoạn đếm thêm: [[1, 2]]  cnt={0: 1, 1: 1, 3: 1}
i=2 x= 1 s=  4 cần s-k=  1 cnt[cần]=1 -> ans=2  đoạn đếm thêm: [[2, 1]]  cnt={0: 1, 1: 1, 3: 1, 4: 1}
i=3 x= 2 s=  6 cần s-k=  3 cnt[cần]=1 -> ans=3  đoạn đếm thêm: [[1, 2]]  cnt={0: 1, 1: 1, 3: 1, 4: 1, 6: 1}
i=4 x= 1 s=  7 cần s-k=  4 cnt[cần]=1 -> ans=4  đoạn đếm thêm: [[2, 1]]  cnt={0: 1, 1: 1, 3: 1, 4: 1, 6: 1, 7: 1}
Bài 2 OK: 4 [[1, 2], [2, 1], [1, 2], [2, 1]]


### Đáp án Bài 3

In [38]:
def exercise_3_sub_solution(nums: List[int], k: int) -> int:
    cnt = {0: 1}
    s = 0
    ans = 0
    for x in nums:
        s += x
        ans += cnt.get(s - k, 0)
        cnt[s] = cnt.get(s, 0) + 1
    return ans


assert exercise_3_sub_solution([1, 1, 1], 2) == 2
assert exercise_3_sub_solution([1, 2, 3], 3) == 2
assert exercise_3_sub_solution([0, 0, 0], 0) == 6
rng = random.Random(3)
for _ in range(200):
    arr = [rng.randint(-3, 3) for _ in range(rng.randint(1, 12))]
    kk = rng.randint(-5, 5)
    assert exercise_3_sub_solution(arr, kk) == sub_cubic(arr, kk)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [39]:
def exercise_4_longest_solution(nums: List[int], k: int) -> int:
    first = {0: -1}
    s = 0
    best = 0
    for i, x in enumerate(nums):
        s += x
        if s - k in first:
            best = max(best, i - first[s - k])
        if s not in first:
            first[s] = i
    return best


def longest_oracle(nums, k):
    best = 0
    for i in range(len(nums)):
        for j in range(i, len(nums)):
            if sum(nums[i:j + 1]) == k:
                best = max(best, j - i + 1)
    return best


assert exercise_4_longest_solution([1, -1, 5, -2, 3], 3) == 4
assert exercise_4_longest_solution([-2, -1, 2, 1], 1) == 2
assert exercise_4_longest_solution([1, 2, 3], 100) == 0
rng = random.Random(4)
for _ in range(300):
    arr = [rng.randint(-3, 3) for _ in range(rng.randint(1, 12))]
    kk = rng.randint(-5, 5)
    assert exercise_4_longest_solution(arr, kk) == longest_oracle(arr, kk)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [40]:
def exercise_5_div_solution(nums: List[int], K: int) -> int:
    cnt = Counter({0: 1})
    s = ans = 0
    for x in nums:
        s = (s + x) % K
        ans += cnt[s]
        cnt[s] += 1
    return ans


def div_oracle(nums, K):
    return sum(1 for i in range(len(nums)) for j in range(i, len(nums)) if sum(nums[i:j + 1]) % K == 0)


assert exercise_5_div_solution([4, 5, 0, -2, -3, 1], 5) == 7
assert exercise_5_div_solution([5], 9) == 0
assert exercise_5_div_solution([-1, 2, 9], 2) == 2
rng = random.Random(5)
for _ in range(300):
    arr = [rng.randint(-6, 6) for _ in range(rng.randint(1, 12))]
    KK = rng.randint(1, 7)
    assert exercise_5_div_solution(arr, KK) == div_oracle(arr, KK)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
from collections import Counter
from typing import List

class Solution:
    def subarraySum(self, nums: List[int], k: int) -> int:
        cnt = Counter({0: 1})      # mốc rỗng
        ans = s = 0
        for x in nums:
            s += x                 # tổng tiền tố hiện tại
            ans += cnt[s - k]      # hỏi trước: có bao nhiêu mốc trước bằng s - k?
            cnt[s] += 1            # ghi sau
        return ans
```

Câu thần chú: **"mốc rỗng `{0: 1}`; hỏi `s - k` trước, ghi `s` sau; số âm thì dùng prefix + map, đừng dùng cửa sổ trượt."**

### Checklist trước khi submit

- [ ] Có `cnt = Counter({0: 1})` (hoặc `{0: 1}`) ngay từ đầu.
- [ ] Thứ tự đúng: `s += x` → `ans += cnt[s - k]` → `cnt[s] += 1`.
- [ ] Dùng **đếm số lần** (Counter/dict), không dùng `set`.
- [ ] Đọc khóa chưa có không báo lỗi (`Counter` hoặc `.get(..., 0)`).
- [ ] Không dùng cửa sổ trượt vì có số âm/0.
- [ ] Test: `[0,0,0]` với `k = 0` (đáp án `6`), `[1]` với `k = 0` (đáp án `0`), `k` âm, mảng 1 phần tử.
- [ ] Skeleton đúng: `class Solution`, method `subarraySum(self, nums: List[int], k: int) -> int`; độ phức tạp `O(n)` thời gian, `O(n)` bộ nhớ.